In [ ]:
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y


Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [80.4 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,966 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,790 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,243 kB]
Hit:13 http://archive.ubuntu.com

In [ ]:
!/usr/local/py310/bin/python -m pip install --upgrade pip
!/usr/local/py310/bin/python -m pip install TTS==0.22.0



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 22.0.2
    Uninstalling pip-22.0.2:
      Successfully uninstalled pip-22.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 150.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 144.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ...

In [ ]:
!/usr/local/py310/bin/python -c "import TTS; print(TTS.__version__)"


0.22.0


In [ ]:
!python --version

Python 3.12.11


In [ ]:
!/usr/local/py310/bin/python -m pip install torch gradio moviepy torchaudio librosa openai-whisper dtw resemblyzer deep_translator numpy==1.22
!/usr/local/py310/bin/python -m pip install transformers==4.41.0
!/usr/local/py310/bin/python -m pip install sentence-transformers==2.2.2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 28.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of moviepy to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 69.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

##Sample with only original audio

In [ ]:
%%writefile vocasync.py
# ============================ IMPORTS ============================
# ============================ IMPORTS ============================
import gradio as gr
import os
import tempfile
import shutil
import uuid
import torch
import librosa
import numpy as np
import soundfile as sf
import gc
from pathlib import Path
from TTS.api import TTS
# Media processing
from moviepy.editor import VideoFileClip, AudioFileClip
from pydub import AudioSegment

# AI Models & Utils
import whisper
from deep_translator import GoogleTranslator
from scipy.spatial.distance import cosine
from resemblyzer import VoiceEncoder, preprocess_wav

# ==== XTTS GLOBAL FIX FOR PYTORCH SAFE UNPICKLING ====
from TTS.tts.configs.xtts_config import XttsConfig, XttsArgs
from TTS.tts.models.xtts import XttsAudioConfig
from TTS.config.shared_configs import BaseDatasetConfig

torch.serialization.add_safe_globals([
    XttsConfig,
    XttsAudioConfig,
    BaseDatasetConfig,
    XttsArgs
])
# =================================================================

# ==== GLOBAL CONFIG & MODEL LOADING ====
print("Setting up global configurations and loading models...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
TEMP_DIR = "temp_outputs"
os.makedirs(TEMP_DIR, exist_ok=True)

print("Loading TTS models (this may take a moment)...")
tts_models = {
    "YourTTS": TTS(model_name="tts_models/multilingual/multi-dataset/your_tts", progress_bar=False).to(device),
    "XTTS": TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device),
}
print("TTS models loaded successfully.")

# ============================ HELPER FUNCTIONS ============================
def clear_gpu_memory(): gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
def preprocess_audio_for_resemblyzer(p): return preprocess_wav(Path(p))
def get_speaker_embedding(p, e):
    try: return e.embed_utterance(preprocess_audio_for_resemblyzer(p))
    except Exception as err: print(f"Embedding Error: {err}"); return None
def calculate_speaker_similarity(p1, p2):
    try:
        encoder = VoiceEncoder(device=device)
        e1, e2 = get_speaker_embedding(p1, encoder), get_speaker_embedding(p2, encoder)
        return (1 - cosine(e1, e2)) if e1 is not None and e2 is not None else None
    except Exception as e: print(f"Similarity Error: {e}"); return None
    finally: del encoder; clear_gpu_memory()
def replace_audio_in_video(v_path, a_path, o_path):
    try:
        video, audio = VideoFileClip(v_path), AudioFileClip(a_path)
        if audio.duration > video.duration: audio = audio.subclip(0, video.duration)
        final = video.set_audio(audio)
        final.write_videofile(o_path, codec="libx264", audio_codec="aac", logger=None, threads=4)
    finally:
        if 'video' in locals(): video.close()
        if 'audio' in locals(): audio.close()
        if 'final' in locals(): final.close()
def pad_audio_to_duration(a_path, target_ms):
    seg = AudioSegment.from_wav(a_path)
    pad_ms = target_ms - len(seg)
    return (seg + AudioSegment.silent(duration=pad_ms)) if pad_ms > 0 else seg[:target_ms]

# ============================ MAIN PROCESSING FUNCTIONS ============================
def split_text_into_chunks(text, max_chars=500):
    """
    Splits text into smaller chunks without breaking words.
    max_chars ~ proportional to XTTS token limit.
    """
    words = text.split()
    chunks, current_chunk = [], []

    for word in words:
        if sum(len(w) for w in current_chunk) + len(word) + len(current_chunk) > max_chars:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
        current_chunk.append(word)

    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks


def process_simple_cloning(
    source_audio,
    source_language,target_language,
    whisper_model_size,
    tts_model_choice,
    progress=gr.Progress()):
    if not source_audio:
        raise gr.Error("❌ Source Audio is required.")


    # --- Transcription & Language Detection ---
    progress(0.1, desc="Loading Whisper model...")
    whisper_model = whisper.load_model(whisper_model_size, device=device)

    progress(0.2, desc="Transcribing & detecting language...")
    transcription = whisper_model.transcribe(source_audio,language=source_language, fp16=torch.cuda.is_available())


    segments = transcription["segments"]
    tts_model = tts_models[tts_model_choice]

    del whisper_model
    clear_gpu_memory()

    # --- Handle Translation Option ---

    temp_dir = tempfile.mkdtemp()
    progress(0.35, desc=f"Translating from {source_language} to {target_language}...")
    generated_segments = []


    for seg in segments:
        start, end, text = seg["start"], seg["end"], seg["text"]
        duration = end - start

        # Translate
        translated_text = GoogleTranslator(source="auto", target=target_language).translate(text)

        # TTS
        out_wav = os.path.join(temp_dir, f"seg_{seg['id']}.wav")
        tts_model.tts_to_file(
            text=translated_text,
            file_path=out_wav,
            speaker_wav=source_audio,
            language=target_language
        )

        # Adjust length (pad or trim)
        speech, sr = librosa.load(out_wav, sr=16000)


        aligned_path = os.path.join(temp_dir, f"aligned_{seg['id']}.wav")
        sf.write(aligned_path, speech, sr)
        generated_segments.append(aligned_path)

    # 3. Concatenate all segments into final audio
    final_audio_path = os.path.join(temp_dir, "final_dub.wav")
    final_audio = np.array([], dtype=np.float32)
    sr = 16000  # use consistent sample rate

    for seg_path in generated_segments:
        seg_audio, _ = librosa.load(seg_path, sr=sr)
        final_audio = np.concatenate([final_audio, seg_audio])

    # Save final
    sf.write(final_audio_path, final_audio, sr)

    return transcription['text'], os.path.abspath(final_audio_path)


# ============================ GRADIO UI (VERSION-COMPATIBLE & SIMPLIFIED) ============================
with gr.Blocks(theme=gr.themes.Soft(), title="VocaSync Suite") as ui:
    gr.Markdown("# 🗣 VocaSync Suite: Voice Cloning & Video Dubbing")
    gr.Markdown("A comprehensive tool for voice cloning, emotion transfer, and AI-powered video dubbing, powered by Coqui TTS.")

    with gr.Tabs():
        with gr.TabItem("🎤 Voice & Emotion Cloning"):
            gr.Markdown("### Create a standard voice clone and an emotion-transferred clone.")

            # ---------- Row 1 ----------
            with gr.Row():
                with gr.Column(scale=1):
                    Audio_input = gr.Audio(label="🎬 Input Audio", type="filepath")


                with gr.Column(scale=1):
                    with gr.Row():
                        source_language = gr.Dropdown(
                            label="Source Language",
                            choices=['hi','en','te','ta','ur'],
                            value='hi'
                        )

                        target_language = gr.Dropdown(
                            label="Target Language",
                            choices=['en'],
                            value='en'
                        )
                    with gr.Row():
                       whisper_model_simple = gr.Dropdown(
                    label="Whisper Model",
                    choices=["tiny", "base", "small", "medium"],
                    value="base"
                )

                       tts_model_simple = gr.Dropdown(
                    label="TTS Model",
                    choices=["YourTTS", "XTTS"],
                    value="XTTS"
                )
                    with gr.Row():
                      clone_btn_simple = gr.Button("🚀 Generate Dub", variant="primary")




            # ---------- Row  ----------
            with gr.Row():
                transcribed_text_simple = gr.Textbox(
                    label="📝 Transcribed Text", interactive=False, lines=4
                )
                final_dub_output = gr.Audio(label="🎧 Output Audio")
            clone_btn_simple.click(
                          fn=process_simple_cloning,
                          inputs=[Audio_input, source_language, target_language, whisper_model_simple, tts_model_simple],
                          outputs=[transcribed_text_simple, final_dub_output]
                      )

if __name__ == "__main__":
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
    os.makedirs(TEMP_DIR, exist_ok=True)
    ui.launch(share=True, debug=True)

Writing vocasync.py


In [ ]:
!/usr/local/py310/bin/python vocasync.py


Setting up global configurations and loading models...
Using device: cuda
Loading TTS models (this may take a moment)...
 > tts_models/multilingual/multi-dataset/your_tts is already downloaded.
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Model fully restored. 
 > Setting up 

## Final with reference audio integration


In [ ]:
%%writefile vocasyncemo.py
import gradio as gr
import os
import tempfile
import shutil
import uuid
import torch
import librosa
import numpy as np
import soundfile as sf
import gc
from pathlib import Path
from TTS.api import TTS
# Media processing
from moviepy.editor import VideoFileClip, AudioFileClip
from pydub import AudioSegment

# AI Models & Utils
import whisper
from deep_translator import GoogleTranslator
from scipy.spatial.distance import cosine
from resemblyzer import VoiceEncoder, preprocess_wav

# ==== XTTS GLOBAL FIX FOR PYTORCH SAFE UNPICKLING ====
from TTS.tts.configs.xtts_config import XttsConfig, XttsArgs
from TTS.tts.models.xtts import XttsAudioConfig
from TTS.config.shared_configs import BaseDatasetConfig

torch.serialization.add_safe_globals([
    XttsConfig,
    XttsAudioConfig,
    BaseDatasetConfig,
    XttsArgs
])
# =================================================================

# ==== GLOBAL CONFIG & MODEL LOADING ====
print("Setting up global configurations and loading models...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
TEMP_DIR = "temp_outputs"
os.makedirs(TEMP_DIR, exist_ok=True)

print("Loading TTS models (this may take a moment)...")
tts_models = {
    "YourTTS": TTS(model_name="tts_models/multilingual/multi-dataset/your_tts", progress_bar=False).to(device),
    "XTTS": TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device),
}
print("TTS models loaded successfully.")

# ============================ HELPER FUNCTIONS ============================
def clear_gpu_memory(): gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
def preprocess_audio_for_resemblyzer(p): return preprocess_wav(Path(p))
def get_speaker_embedding(p, e):
    try: return e.embed_utterance(preprocess_audio_for_resemblyzer(p))
    except Exception as err: print(f"Embedding Error: {err}"); return None

def calculate_speaker_similarity(original_audio_path: str, cloned_audio_path: str):
    try:
        encoder = VoiceEncoder(device=device)
        emb1 = get_speaker_embedding(original_audio_path, encoder)
        emb2 = get_speaker_embedding(cloned_audio_path, encoder)
        if emb1 is None or emb2 is None:
            return None
        return 1 - cosine(emb1, emb2)
    except Exception as e:
        print("Similarity Error:", e)
        return None
def process_simple_cloning(
    video_file,
    reference_audio,
    source_language,
    target_language,
    whisper_model_name,
    tts_model_name,
    temp_dir="temp_segments",
    emotion_temp_dir="temp_emotion_segments"
):
    # Save video
    original_filename = os.path.basename(video_file)
    local_video_path = os.path.join(tempfile.gettempdir(), original_filename)
    shutil.copy(video_file, local_video_path)

    # Extract Audio
    clip = VideoFileClip(local_video_path)
    audio_filename = os.path.splitext(original_filename)[0] + ".wav"
    source_audio = os.path.join(TEMP_DIR, audio_filename)
    clip.audio.write_audiofile(source_audio, logger=None)
    clip.close()

    os.makedirs(temp_dir, exist_ok=True)
    os.makedirs(emotion_temp_dir, exist_ok=True)
    whisper_model = whisper.load_model(whisper_model_name)
    tts_model=tts_models[tts_model_name]
    # 1. Transcribe with timestamps
    transcription = whisper_model.transcribe(source_audio, language=source_language, fp16=False)
    segments = transcription["segments"]
    final_translated_text=''
    sr = 16000
    final_audio = np.array([], dtype=np.float32)
    emotion_final_audio = np.array([], dtype=np.float32)
    current_time = 0.0
    emotion_current_time = 0.0

    for seg in segments:
        start, end, text = seg["start"], seg["end"], seg["text"]

        # --- Insert silence for pauses before this segment ---
        if start > current_time:
            silence_duration = start - current_time
            silence = np.zeros(int(silence_duration * sr), dtype=np.float32)
            final_audio = np.concatenate([final_audio, silence])

        if start > emotion_current_time:
            silence_duration = start - emotion_current_time
            silence = np.zeros(int(silence_duration * sr), dtype=np.float32)
            emotion_final_audio = np.concatenate([emotion_final_audio, silence])

        # --- Translate ---
        translated_text = GoogleTranslator(source="auto", target=target_language).translate(text)
        final_translated_text += translated_text + ' '
        # --- TTS (original speaker clone) ---
        out_wav = os.path.join(temp_dir, f"seg_{seg['id']}.wav")
        tts_model.tts_to_file(
            text=translated_text,
            file_path=out_wav,
            speaker_wav=source_audio,
            language=target_language,
        )
        speech, _ = librosa.load(out_wav, sr=sr)
        final_audio = np.concatenate([final_audio, speech])

        # --- TTS (emotion/reference speaker clone) ---
        emotion_out_wav = os.path.join(emotion_temp_dir, f"seg_{seg['id']}.wav")
        tts_model.tts_to_file(
            text=translated_text,
            file_path=emotion_out_wav,
            speaker_wav=reference_audio,
            language=target_language,
        )
        emotion_speech, _ = librosa.load(emotion_out_wav, sr=sr)
        emotion_final_audio = np.concatenate([emotion_final_audio, emotion_speech])

        current_time = end
        emotion_current_time = end

    # 3. Save final dubs
    final_audio_path = os.path.join(temp_dir, "final_dub.wav")
    emotion_final_audio_path = os.path.join(emotion_temp_dir, "emotion_final_dub.wav")

    sf.write(final_audio_path, final_audio, sr)
    sf.write(emotion_final_audio_path, emotion_final_audio, sr)

    similarity_score = calculate_speaker_similarity(source_audio, final_audio_path)
    similarity_text = f"{similarity_score:.4f}" if similarity_score else "⚠ Similarity Error"
    emotion_similarity_score = calculate_speaker_similarity( source_audio,emotion_final_audio_path)
    emotion_similarity_text = f"{emotion_similarity_score:.4f}" if emotion_similarity_score else "⚠ Similarity Error"


    return transcription["text"],final_translated_text, os.path.abspath(final_audio_path), os.path.abspath(emotion_final_audio_path), similarity_text,emotion_similarity_text


# ============================ GRADIO UI (VERSION-COMPATIBLE & SIMPLIFIED) ============================
with gr.Blocks(theme=gr.themes.Soft(), title="VocaSync Suite") as ui:

    with gr.Tabs():
        with gr.TabItem("🎤 Voice & Emotion Cloning"):

            # ---------- Row 1 ----------

            with gr.Row():
                with gr.Column(scale=1):

                    Video_input = gr.File(label="🎥 Upload MP4 Video", type="filepath",scale=2)
                    with gr.Row():
                        source_language = gr.Dropdown(
                            label="Source Language",
                            choices=['hi','en','te','ta','ur'],
                            value='hi'
                        )

                        target_language = gr.Dropdown(
                            label="Target Language",
                            choices=['en'],
                            value='en'
                        )
                    with gr.Row():
                       whisper_model_simple = gr.Dropdown(
                    label="Whisper Model",
                    choices=["tiny", "base", "small", "medium"],
                    value="base"
                )

                       tts_model_simple = gr.Dropdown(
                    label="TTS Model",
                    choices=["YourTTS", "XTTS"],
                    value="XTTS"
                )



                with gr.Column(scale=1):
                    emotion_audio_input = gr.Audio(label="🎬 reference emotion audio", type="filepath")


                    with gr.Row():
                      clone_btn_simple = gr.Button("🚀 Generate Dub", variant="primary")

            # ---------- Row  ----------
            with gr.Row():
                transcribed_text = gr.Textbox(
                    label="📝 Transcribed Text", interactive=False, lines=4
                )
                translated_text= gr.Textbox(
                    label="Translated Text", interactive=False, lines=4
                )

            with gr.Row():
                final_dub_output = gr.Audio(label="🎧 Output Audio")

                emotion_final_dub_output = gr.Audio(label="🎧 Emotion Output Audio")
                with gr.Column(scale=1):
                  similarity_score = gr.Textbox(label="Similarity Score")
                  emotion_similarity_score = gr.Textbox(label="Emotion Similarity Score")
            clone_btn_simple.click(
                          fn=process_simple_cloning,
                          inputs=[Video_input, emotion_audio_input, source_language, target_language, whisper_model_simple, tts_model_simple],
                          outputs=[transcribed_text, translated_text, final_dub_output, emotion_final_dub_output, similarity_score,emotion_similarity_score]
                      )

if __name__ == "__main__":
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
    os.makedirs(TEMP_DIR, exist_ok=True)
    ui.launch(share=True, debug=True)

Writing vocasyncemo.py


In [ ]:
!/usr/local/py310/bin/python vocasyncemo.py


Setting up global configurations and loading models...
Using device: cuda
Loading TTS models (this may take a moment)...
 > tts_models/multilingual/multi-dataset/your_tts is already downloaded.
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Model fully restored. 
 > Setting up 